# Evidence-Grounded Multilingual Taxonomy Evaluator

This notebook trains a **multilingual, single-prompt, evidence-grounded image-editing evaluator**
that keeps the same protocol as the earlier runs:

- split by **original id first**
- then expand each original into **English / Hindi / Bangla** rows
- inference uses **one prompt in one language at a time**
- primary benchmark remains the **taxonomy-positive held-out split**
- the output taxonomy is unchanged

The model is designed to predict taxonomy labels **and** expose structured evidence:
grounding, correspondence, local change, outside-region change, and prompt-operation cues.

In [1]:
# Cell 1 — environment and imports

import os
import re
import gc
import json
import math
import time
import copy
import random
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, average_precision_score

from transformers import AutoTokenizer, XLMRobertaModel, ViTModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
# Cell 2 — CONFIG

from pathlib import Path

# ----------------------------
# Paths
# ----------------------------
IMAGE_ROOT = "/kaggle/input/datasets/tusherbhomik/picobanana-images"
SPLITS_DIR = "/kaggle/input/datasets/tusherbhomik/picobanana-splits"

TRAIN_JSONL = f"{SPLITS_DIR}/train_originals.jsonl"
VAL_JSONL   = f"{SPLITS_DIR}/val_originals.jsonl"
TEST_JSONL  = f"{SPLITS_DIR}/test_originals.jsonl"

# ----------------------------
# Output
# ----------------------------
WORK_DIR = Path("/kaggle/working/evidence_grounded_taxonomy_eval")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Core protocol
# ----------------------------
SEED = 42
LANGS = ("en", "hi", "bn")
TRAIN_NEG_RATIO = 0.35           # include a small clean/no-error set for calibration
PRIMARY_EVAL_MODE = "taxonomy_positive"   # keep comparability with earlier benchmark
SMOKE_TEST = False
SMOKE_LIMIT_ORIGINALS = 120

# ----------------------------
# Model
# ----------------------------
VISION_MODEL_ID = "google/vit-base-patch16-224-in21k"
TEXT_MODEL_ID   = "xlm-roberta-base"
IMG_SIZE = 224
MAX_TEXT_LEN = 128
HIDDEN_DIM = 256
DROPOUT = 0.15
NUM_OP_TYPES = 8

# ----------------------------
# Training
# ----------------------------
BATCH_SIZE = 8
NUM_WORKERS = 0
EPOCHS = 8
PATIENCE = 3
HEAD_LR = 3e-4
ENCODER_LR = 2e-5
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
USE_AMP = True
FOCAL_GAMMA = 1.5
AUX_OP_LOSS_WEIGHT = 0.20

# ----------------------------
# Unfreezing
# ----------------------------
UNFREEZE_TEXT_LAYERS = 2
UNFREEZE_VISION_LAYERS = 2

print("IMAGE_ROOT             :", IMAGE_ROOT)
print("SPLITS_DIR             :", SPLITS_DIR)
print("WORK_DIR               :", WORK_DIR)
print("VISION_MODEL_ID        :", VISION_MODEL_ID)
print("TEXT_MODEL_ID          :", TEXT_MODEL_ID)
print("PRIMARY_EVAL_MODE      :", PRIMARY_EVAL_MODE)
print("TRAIN_NEG_RATIO        :", TRAIN_NEG_RATIO)
print("BATCH_SIZE             :", BATCH_SIZE)
print("EPOCHS                 :", EPOCHS)

IMAGE_ROOT             : /kaggle/input/datasets/tusherbhomik/picobanana-images
SPLITS_DIR             : /kaggle/input/datasets/tusherbhomik/picobanana-splits
WORK_DIR               : /kaggle/working/evidence_grounded_taxonomy_eval
VISION_MODEL_ID        : google/vit-base-patch16-224-in21k
TEXT_MODEL_ID          : xlm-roberta-base
PRIMARY_EVAL_MODE      : taxonomy_positive
TRAIN_NEG_RATIO        : 0.35
BATCH_SIZE             : 8
EPOCHS                 : 8


In [3]:
# Cell 3 — PATH CHECK

checks = [
    ("IMAGE_ROOT", IMAGE_ROOT),
    ("source images", f"{IMAGE_ROOT}/images/source"),
    ("target images", f"{IMAGE_ROOT}/images/target"),
    ("TRAIN_JSONL", TRAIN_JSONL),
    ("VAL_JSONL", VAL_JSONL),
    ("TEST_JSONL", TEST_JSONL),
]

print("── Path checks ─────────────────────────────────────────────────────────")
all_ok = True
for name, path in checks:
    ok = Path(path).exists()
    print(f"[{'OK' if ok else 'MISSING'}] {name:<16} -> {path}")
    all_ok = all_ok and ok

if not all_ok:
    raise FileNotFoundError("One or more required paths are missing. Fix Cell 2 first.")

print("\nAll required paths look OK.")

── Path checks ─────────────────────────────────────────────────────────
[OK] IMAGE_ROOT       -> /kaggle/input/datasets/tusherbhomik/picobanana-images
[OK] source images    -> /kaggle/input/datasets/tusherbhomik/picobanana-images/images/source
[OK] target images    -> /kaggle/input/datasets/tusherbhomik/picobanana-images/images/target
[OK] TRAIN_JSONL      -> /kaggle/input/datasets/tusherbhomik/picobanana-splits/train_originals.jsonl
[OK] VAL_JSONL        -> /kaggle/input/datasets/tusherbhomik/picobanana-splits/val_originals.jsonl
[OK] TEST_JSONL       -> /kaggle/input/datasets/tusherbhomik/picobanana-splits/test_originals.jsonl

All required paths look OK.


In [4]:
# Cell 4 — taxonomy constants, prompt operations, and data helpers

ERROR_TYPES = [
    "Wrong Object",
    "Missing Object",
    "Extra Object",
    "Wrong Attribute",
    "Spatial Error",
    "Style Mismatch",
    "Over-editing",
    "Under-editing",
    "Artifact / Quality Issue",
    "Ambiguous Prompt",
    "Failed Removal",
]
NUM_CLASSES = len(ERROR_TYPES)
ERROR_TO_IDX = {e: i for i, e in enumerate(ERROR_TYPES)}

OP_TYPES = [
    "remove",
    "add",
    "replace",
    "recolor",
    "style",
    "move",
    "background",
    "other",
]
OP_TO_IDX = {op: i for i, op in enumerate(OP_TYPES)}

print(f"{NUM_CLASSES} taxonomy labels loaded.")
print(f"{len(OP_TYPES)} prompt operation labels loaded.")


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_jsonl(path: str) -> List[Dict[str, Any]]:
    p = Path(path)
    if not p.exists():
        return []
    return [json.loads(line) for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]


def save_json(path: str | Path, obj: Dict[str, Any]) -> None:
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


def normalize_path(p: str) -> str:
    return str(p).replace("\\", "/").lstrip("./")


def labels_to_multihot(labels: List[str]) -> List[float]:
    vec = [0.0] * NUM_CLASSES
    for lbl in labels:
        if lbl in ERROR_TO_IDX:
            vec[ERROR_TO_IDX[lbl]] = 1.0
    return vec


def infer_prompt_operation(prompt: str) -> str:
    p = (prompt or "").strip().lower()

    patterns = {
        "remove": [
            r"\bremove\b", r"\bdelete\b", r"\berase\b", r"\bwithout\b",
            r"हटा", r"निकाल", r"मिटा",
            r"মুছে", r"সরিয়ে", r"সরিয়ে", r"বাদ দাও",
        ],
        "add": [
            r"\badd\b", r"\binsert\b", r"\binclude\b", r"\bput\b",
            r"जोड़", r"डाल", r"যোগ", r"দাও",
        ],
        "replace": [
            r"\breplace\b", r"\bswap\b", r"\bchange the .* to\b",
            r"बदल", r"পরিবর্তন", r"বদল",
        ],
        "recolor": [
            r"\bcolor\b", r"\brecolor\b", r"\bturn .* (red|blue|green|black|white|yellow|pink|orange|purple|brown|gold|silver)\b",
            r"रंग", r"কালো", r"লাল", r"নীল", r"সাদা",
        ],
        "style": [
            r"\bstyle\b", r"\bmake it look like\b", r"\bcartoon\b", r"\boil painting\b",
            r"शैली", r"স্টাইল", r"ধাঁচ",
        ],
        "move": [
            r"\bmove\b", r"\bshift\b", r"\bplace\b", r"\bput .* on the left\b", r"\bput .* on the right\b",
            r"स्थान", r"बाएँ", r"दाएँ", r"সরে", r"বামে", r"ডানে",
        ],
        "background": [
            r"\bbackground\b", r"\bsky\b", r"\bwall\b", r"\bscene\b",
            r"पृष्ठभूमि", r"आसमान", r"ব্যাকগ্রাউন্ড", r"আকাশ",
        ],
    }

    for op, pats in patterns.items():
        for pat in pats:
            if re.search(pat, p):
                return op
    return "other"


def expand_rows_by_language(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    expanded = []
    for row in rows:
        labels = [str(x) for x in row.get("taxonomy_labels", [])]
        for lang in LANGS:
            text = str(row.get(f"instruction_{lang}") or "").strip()
            if not text:
                continue
            op_type = infer_prompt_operation(text)
            expanded.append({
                "id": str(row["id"]),
                "lang": lang,
                "instruction": text,
                "source_path": normalize_path(str(row.get("source_path") or "")),
                "target_path": normalize_path(str(row.get("target_path") or "")),
                "taxonomy_labels": labels,
                "target_vec": labels_to_multihot(labels),
                "has_error": 1 if len(labels) > 0 else 0,
                "op_type": op_type,
                "op_type_id": OP_TO_IDX[op_type],
            })
    return expanded


def filter_taxonomy_positive(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    return [r for r in rows if r.get("taxonomy_labels") and len(r["taxonomy_labels"]) > 0]


def build_train_rows(train_orig: List[Dict[str, Any]], neg_ratio: float = 0.35) -> List[Dict[str, Any]]:
    pos = filter_taxonomy_positive(train_orig)
    neg = [r for r in train_orig if not r.get("taxonomy_labels")]
    if neg_ratio > 0 and len(pos) > 0 and len(neg) > 0:
        keep_neg = min(len(neg), int(round(len(pos) * neg_ratio)))
        rng = random.Random(SEED)
        neg = rng.sample(neg, keep_neg)
    return expand_rows_by_language(pos + neg)


seed_everything(SEED)

11 taxonomy labels loaded.
8 prompt operation labels loaded.


In [5]:
# Cell 5 — load splits and build multilingual rows

train_orig = load_jsonl(TRAIN_JSONL)
val_orig   = load_jsonl(VAL_JSONL)
test_orig  = load_jsonl(TEST_JSONL)

if SMOKE_TEST:
    train_orig = train_orig[:SMOKE_LIMIT_ORIGINALS]
    val_orig   = val_orig[:max(30, SMOKE_LIMIT_ORIGINALS // 8)]
    test_orig  = test_orig[:max(30, SMOKE_LIMIT_ORIGINALS // 8)]

print(f"Loaded originals — train: {len(train_orig)}, val: {len(val_orig)}, test: {len(test_orig)}")

train_rows = build_train_rows(train_orig, neg_ratio=TRAIN_NEG_RATIO)
val_rows_full = expand_rows_by_language(val_orig)
test_rows_full = expand_rows_by_language(test_orig)

val_rows_benchmark = expand_rows_by_language(filter_taxonomy_positive(val_orig))
test_rows_benchmark = expand_rows_by_language(filter_taxonomy_positive(test_orig))

print(f"Train rows (multilingual, with sampled negatives): {len(train_rows)}")
print(f"Val rows full                              : {len(val_rows_full)}")
print(f"Test rows full                             : {len(test_rows_full)}")
print(f"Val rows benchmark (taxonomy-positive)     : {len(val_rows_benchmark)}")
print(f"Test rows benchmark (taxonomy-positive)    : {len(test_rows_benchmark)}")

dataset_summary = {
    "train_originals": len(train_orig),
    "val_originals": len(val_orig),
    "test_originals": len(test_orig),
    "train_rows": len(train_rows),
    "val_rows_full": len(val_rows_full),
    "test_rows_full": len(test_rows_full),
    "val_rows_benchmark": len(val_rows_benchmark),
    "test_rows_benchmark": len(test_rows_benchmark),
    "languages": list(LANGS),
    "train_neg_ratio": TRAIN_NEG_RATIO,
}
save_json(WORK_DIR / "dataset_summary.json", dataset_summary)

Loaded originals — train: 4976, val: 621, test: 597
Train rows (multilingual, with sampled negatives): 9768
Val rows full                              : 1863
Test rows full                             : 1791
Val rows benchmark (taxonomy-positive)     : 912
Test rows benchmark (taxonomy-positive)    : 813


In [6]:
# Cell 6 — paired transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

EVAL_TRANSFORM = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print("Eval transform ready.")

Eval transform ready.


In [7]:
# Cell 7 — dataset class

baseline_tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_ID)

class EvidenceTaxonomyDataset(Dataset):
    """
    Multilingual single-prompt rows:
    (source image, edited image, one prompt) -> taxonomy + evidence auxiliaries
    """
    def __init__(
        self,
        rows: List[Dict[str, Any]],
        image_root: str,
        tokenizer,
        max_text_len: int = 128,
        image_transform=None,
    ):
        self.rows = rows
        self.image_root = image_root
        self.tokenizer = tokenizer
        self.max_text_len = max_text_len
        self.image_transform = image_transform

    def __len__(self) -> int:
        return len(self.rows)

    def _load_image(self, rel_path: str) -> torch.Tensor:
        abs_path = os.path.join(self.image_root, rel_path)
        img = Image.open(abs_path).convert("RGB")
        try:
            if self.image_transform is not None:
                img = self.image_transform(img)
            return img
        finally:
            try:
                if hasattr(img, "close"):
                    img.close()
            except Exception:
                pass

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.rows[idx]
        src = self._load_image(row["source_path"])
        tgt = self._load_image(row["target_path"])

        tok = self.tokenizer(
            row["instruction"],
            truncation=True,
            padding="max_length",
            max_length=self.max_text_len,
            return_tensors="pt",
        )

        return {
            "id": row["id"],
            "lang": row["lang"],
            "instruction": row["instruction"],
            "src_img": src,
            "tgt_img": tgt,
            "input_ids": tok["input_ids"].squeeze(0),
            "attention_mask": tok["attention_mask"].squeeze(0),
            "target": torch.tensor(row["target_vec"], dtype=torch.float32),
            "has_error": torch.tensor(float(row["has_error"]), dtype=torch.float32),
            "op_type_id": torch.tensor(int(row["op_type_id"]), dtype=torch.long),
        }

print("Dataset class ready.")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Dataset class ready.


In [8]:
# Cell 8 — evidence-grounded model architecture

def freeze_all(module: nn.Module) -> None:
    for p in module.parameters():
        p.requires_grad = False


def unfreeze_last_n_layers(module: nn.Module, n: int) -> None:
    if n <= 0:
        return
    layers = None
    if hasattr(module, "encoder") and hasattr(module.encoder, "layer"):
        layers = module.encoder.layer
    if layers is None:
        return
    for layer in layers[-n:]:
        for p in layer.parameters():
            p.requires_grad = True


class MLP(nn.Module):
    def __init__(self, dim_in: int, dim_hidden: int, dim_out: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim_in, dim_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_hidden, dim_out),
        )

    def forward(self, x):
        return self.net(x)


class EvidenceGroundedTaxonomyModel(nn.Module):
    def __init__(
        self,
        vision_model_id: str,
        text_model_id: str,
        hidden_dim: int,
        num_classes: int,
        num_op_types: int,
        dropout: float = 0.1,
        unfreeze_text_layers: int = 2,
        unfreeze_vision_layers: int = 2,
    ):
        super().__init__()
        self.vision = ViTModel.from_pretrained(vision_model_id)
        self.text = XLMRobertaModel.from_pretrained(text_model_id)

        freeze_all(self.vision)
        freeze_all(self.text)
        unfreeze_last_n_layers(self.vision, unfreeze_vision_layers)
        unfreeze_last_n_layers(self.text, unfreeze_text_layers)

        vision_dim = self.vision.config.hidden_size
        text_dim = self.text.config.hidden_size

        self.patch_proj = nn.Linear(vision_dim, hidden_dim)
        self.vision_cls_proj = nn.Linear(vision_dim, hidden_dim)
        self.text_cls_proj = nn.Linear(text_dim, hidden_dim)
        self.text_query_proj = nn.Linear(hidden_dim, hidden_dim)

        self.src_presence_head = MLP(hidden_dim * 3, hidden_dim, 1, dropout)
        self.tgt_presence_head = MLP(hidden_dim * 3, hidden_dim, 1, dropout)
        self.local_change_head = MLP(hidden_dim, hidden_dim, 1, dropout)
        self.global_change_head = MLP(hidden_dim, hidden_dim, 1, dropout)
        self.outside_change_head = MLP(hidden_dim, hidden_dim, 1, dropout)
        self.op_head = MLP(hidden_dim, hidden_dim, num_op_types, dropout)

        evidence_dim = (
            hidden_dim * 8
            + 7  # scalar evidence
            + num_op_types
        )

        self.taxonomy_head = nn.Sequential(
            nn.Linear(evidence_dim, hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, src_img, tgt_img, input_ids, attention_mask):
        src_out = self.vision(pixel_values=src_img)
        tgt_out = self.vision(pixel_values=tgt_img)
        txt_out = self.text(input_ids=input_ids, attention_mask=attention_mask)

        src_seq = src_out.last_hidden_state
        tgt_seq = tgt_out.last_hidden_state
        txt_seq = txt_out.last_hidden_state

        src_cls = self.vision_cls_proj(src_seq[:, 0])
        tgt_cls = self.vision_cls_proj(tgt_seq[:, 0])
        src_patches = self.patch_proj(src_seq[:, 1:])
        tgt_patches = self.patch_proj(tgt_seq[:, 1:])
        txt_cls = self.text_cls_proj(txt_seq[:, 0])

        # Prompt-conditioned grounding
        q = F.normalize(self.text_query_proj(txt_cls), dim=-1)
        src_norm = F.normalize(src_patches, dim=-1)
        tgt_norm = F.normalize(tgt_patches, dim=-1)

        src_scores = torch.einsum("bnd,bd->bn", src_norm, q) / math.sqrt(src_norm.shape[-1])
        tgt_scores = torch.einsum("bnd,bd->bn", tgt_norm, q) / math.sqrt(tgt_norm.shape[-1])

        src_attn = torch.softmax(src_scores, dim=-1)
        tgt_attn = torch.softmax(tgt_scores, dim=-1)

        src_local = torch.sum(src_patches * src_attn.unsqueeze(-1), dim=1)
        tgt_local = torch.sum(tgt_patches * tgt_attn.unsqueeze(-1), dim=1)

        patch_diff = torch.abs(src_patches - tgt_patches)
        local_diff = torch.sum(patch_diff * src_attn.unsqueeze(-1), dim=1)

        outside_w = (1.0 - src_attn).clamp(min=1e-6)
        outside_w = outside_w / outside_w.sum(dim=-1, keepdim=True)
        outside_diff = torch.sum(patch_diff * outside_w.unsqueeze(-1), dim=1)

        # Correspondence: does the source-local target still align to a region in the edited image?
        corr_scores = torch.einsum(
            "bnd,bd->bn",
            F.normalize(tgt_patches, dim=-1),
            F.normalize(src_local, dim=-1),
        )
        corr_max = corr_scores.max(dim=-1).values.unsqueeze(-1)
        corr_mean = corr_scores.mean(dim=-1, keepdim=True)

        src_presence = torch.sigmoid(self.src_presence_head(torch.cat([txt_cls, src_local, src_cls], dim=-1)))
        tgt_presence = torch.sigmoid(self.tgt_presence_head(torch.cat([txt_cls, tgt_local, tgt_cls], dim=-1)))
        local_change_score = torch.sigmoid(self.local_change_head(local_diff))
        global_change_feat = torch.abs(src_cls - tgt_cls)
        global_change_score = torch.sigmoid(self.global_change_head(global_change_feat))
        outside_change_score = torch.sigmoid(self.outside_change_head(outside_diff))
        op_logits = self.op_head(txt_cls)
        op_probs = torch.softmax(op_logits, dim=-1)

        evidence_vec = torch.cat([
            txt_cls,
            src_cls,
            tgt_cls,
            src_local,
            tgt_local,
            global_change_feat,
            local_diff,
            outside_diff,
            src_presence,
            tgt_presence,
            corr_max,
            corr_mean,
            local_change_score,
            global_change_score,
            outside_change_score,
            op_probs,
        ], dim=-1)

        taxonomy_logits = self.taxonomy_head(evidence_vec)

        evidence = {
            "src_attn": src_attn,
            "tgt_attn": tgt_attn,
            "src_presence": src_presence.squeeze(-1),
            "tgt_presence": tgt_presence.squeeze(-1),
            "correspondence_max": corr_max.squeeze(-1),
            "correspondence_mean": corr_mean.squeeze(-1),
            "local_change_score": local_change_score.squeeze(-1),
            "global_change_score": global_change_score.squeeze(-1),
            "outside_change_score": outside_change_score.squeeze(-1),
            "op_logits": op_logits,
            "op_probs": op_probs,
        }
        return taxonomy_logits, evidence


model = EvidenceGroundedTaxonomyModel(
    vision_model_id=VISION_MODEL_ID,
    text_model_id=TEXT_MODEL_ID,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    num_op_types=len(OP_TYPES),
    dropout=DROPOUT,
    unfreeze_text_layers=UNFREEZE_TEXT_LAYERS,
    unfreeze_vision_layers=UNFREEZE_VISION_LAYERS,
).to(device)

print("Evidence-grounded model ready.")

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evidence-grounded model ready.


In [9]:
# Cell 9 — losses, metrics, threshold tuning, and optimizer

class FocalBCEWithLogitsLoss(nn.Module):
    def __init__(self, pos_weight: Optional[torch.Tensor] = None, gamma: float = 1.5):
        super().__init__()
        self.pos_weight = pos_weight
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none",
            pos_weight=self.pos_weight,
        )
        probs = torch.sigmoid(logits)
        pt = targets * probs + (1 - targets) * (1 - probs)
        focal = (1 - pt).pow(self.gamma)
        return (focal * bce).mean()


def compute_pos_weight(rows: List[Dict[str, Any]], device):
    y = np.array([r["target_vec"] for r in rows], dtype=np.float32)
    pos = y.sum(axis=0)
    neg = len(y) - pos
    pos_weight = np.clip(neg / np.maximum(pos, 1.0), 1.0, 25.0)
    return torch.tensor(pos_weight, dtype=torch.float32, device=device)


def compute_multilabel_metrics(y_true, y_prob, y_pred):
    out = {}
    out["micro_f1"] = float(f1_score(y_true, y_pred, average="micro", zero_division=0))
    out["macro_f1"] = float(f1_score(y_true, y_pred, average="macro", zero_division=0))

    per_class_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
    support_mask = y_true.sum(axis=0) > 0
    out["macro_f1_supported"] = float(per_class_f1[support_mask].mean()) if support_mask.any() else 0.0

    try:
        out["mAP_micro"] = float(average_precision_score(y_true, y_prob, average="micro"))
    except Exception:
        out["mAP_micro"] = 0.0

    try:
        ap_per_class = []
        for i in range(y_true.shape[1]):
            if y_true[:, i].sum() > 0:
                ap_per_class.append(average_precision_score(y_true[:, i], y_prob[:, i]))
        out["mAP_macro_supported"] = float(np.mean(ap_per_class)) if ap_per_class else 0.0
    except Exception:
        out["mAP_macro_supported"] = 0.0
    return out


def compute_per_class_metrics(y_true, y_prob, y_pred) -> pd.DataFrame:
    rows = []
    per_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
    per_prec = precision_score(y_true, y_pred, average=None, zero_division=0)
    per_rec = recall_score(y_true, y_pred, average=None, zero_division=0)
    support = y_true.sum(axis=0)
    for i, cls in enumerate(ERROR_TYPES):
        rows.append({
            "class": cls,
            "f1": float(per_f1[i]),
            "precision": float(per_prec[i]),
            "recall": float(per_rec[i]),
            "support": int(support[i]),
        })
    df = pd.DataFrame(rows)
    print(df)
    return df


def tune_thresholds(y_true: np.ndarray, y_prob: np.ndarray) -> List[float]:
    thresholds = []
    grid = np.round(np.arange(0.15, 0.86, 0.05), 2)
    for i in range(y_true.shape[1]):
        if y_true[:, i].sum() == 0:
            thresholds.append(0.50)
            continue
        best_t, best_f1 = 0.50, -1.0
        for t in grid:
            y_pred_i = (y_prob[:, i] >= t).astype(int)
            f1 = f1_score(y_true[:, i], y_pred_i, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = float(t)
        thresholds.append(best_t)
    return thresholds


def run_eval(model, loader, criterion_tax, criterion_op, device, use_amp=True):
    model.eval()
    all_probs, all_true, all_logits = [], [], []
    all_langs, all_ids, all_pred_labels = [], [], []
    all_evidence_rows = []
    total_loss = 0.0
    n = 0

    with torch.no_grad():
        for batch in loader:
            src = batch["src_img"].to(device, non_blocking=True)
            tgt = batch["tgt_img"].to(device, non_blocking=True)
            ids = batch["input_ids"].to(device, non_blocking=True)
            mask = batch["attention_mask"].to(device, non_blocking=True)
            tgts = batch["target"].to(device, non_blocking=True)
            op_ids = batch["op_type_id"].to(device, non_blocking=True)

            with torch.autocast(device_type="cuda", enabled=(use_amp and device.type == "cuda")):
                logits, evidence = model(src, tgt, ids, mask)
                loss_tax = criterion_tax(logits, tgts)
                loss_op = criterion_op(evidence["op_logits"], op_ids)
                loss = loss_tax + AUX_OP_LOSS_WEIGHT * loss_op

            probs = torch.sigmoid(logits).cpu().numpy()
            total_loss += float(loss.item()) * tgts.shape[0]
            n += int(tgts.shape[0])

            all_probs.append(probs)
            all_true.append(tgts.cpu().numpy())
            all_logits.append(logits.cpu().numpy())
            all_langs.extend(batch["lang"])
            all_ids.extend(batch["id"])

            for i in range(len(batch["id"])):
                all_evidence_rows.append({
                    "id": batch["id"][i],
                    "lang": batch["lang"][i],
                    "src_presence": float(evidence["src_presence"][i].detach().cpu()),
                    "tgt_presence": float(evidence["tgt_presence"][i].detach().cpu()),
                    "correspondence_max": float(evidence["correspondence_max"][i].detach().cpu()),
                    "correspondence_mean": float(evidence["correspondence_mean"][i].detach().cpu()),
                    "local_change_score": float(evidence["local_change_score"][i].detach().cpu()),
                    "global_change_score": float(evidence["global_change_score"][i].detach().cpu()),
                    "outside_change_score": float(evidence["outside_change_score"][i].detach().cpu()),
                    "pred_op_type": OP_TYPES[int(torch.argmax(evidence["op_probs"][i]).detach().cpu())],
                })

    y_prob = np.concatenate(all_probs, axis=0)
    y_true = np.concatenate(all_true, axis=0)
    y_logits = np.concatenate(all_logits, axis=0)
    return total_loss / max(n, 1), y_true, y_prob, y_logits, all_langs, all_ids, pd.DataFrame(all_evidence_rows)


def collect_trainable_params(model):
    head_params, enc_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("vision.") or name.startswith("text."):
            enc_params.append(p)
        else:
            head_params.append(p)
    return head_params, enc_params


pos_weight = compute_pos_weight(train_rows, device)
criterion_tax = FocalBCEWithLogitsLoss(pos_weight=pos_weight, gamma=FOCAL_GAMMA)
criterion_op = nn.CrossEntropyLoss()

head_params, enc_params = collect_trainable_params(model)
optimizer = torch.optim.AdamW([
    {"params": head_params, "lr": HEAD_LR},
    {"params": enc_params, "lr": ENCODER_LR},
], weight_decay=WEIGHT_DECAY)

scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and device.type == "cuda"))

print("Losses and optimizer ready.")

Losses and optimizer ready.


In [10]:
# Cell 10 — dataloaders

train_ds = EvidenceTaxonomyDataset(
    train_rows, IMAGE_ROOT, baseline_tokenizer, MAX_TEXT_LEN, EVAL_TRANSFORM
)
val_ds_full = EvidenceTaxonomyDataset(
    val_rows_full, IMAGE_ROOT, baseline_tokenizer, MAX_TEXT_LEN, EVAL_TRANSFORM
)
test_ds_full = EvidenceTaxonomyDataset(
    test_rows_full, IMAGE_ROOT, baseline_tokenizer, MAX_TEXT_LEN, EVAL_TRANSFORM
)
val_ds_benchmark = EvidenceTaxonomyDataset(
    val_rows_benchmark, IMAGE_ROOT, baseline_tokenizer, MAX_TEXT_LEN, EVAL_TRANSFORM
)
test_ds_benchmark = EvidenceTaxonomyDataset(
    test_rows_benchmark, IMAGE_ROOT, baseline_tokenizer, MAX_TEXT_LEN, EVAL_TRANSFORM
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(device.type=="cuda"))
val_loader_full = DataLoader(val_ds_full, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(device.type=="cuda"))
test_loader_full = DataLoader(test_ds_full, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(device.type=="cuda"))
val_loader_benchmark = DataLoader(val_ds_benchmark, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(device.type=="cuda"))
test_loader_benchmark = DataLoader(test_ds_benchmark, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(device.type=="cuda"))

print("Train batches         :", len(train_loader))
print("Val benchmark batches :", len(val_loader_benchmark))
print("Test benchmark batches:", len(test_loader_benchmark))

Train batches         : 1221
Val benchmark batches : 114
Test benchmark batches: 102


In [11]:
# Cell 11 — train evidence-grounded evaluator

best_score = -1.0
best_state = None
best_thresholds = [0.5] * NUM_CLASSES
history = []
no_improve = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    n = 0

    for batch in train_loader:
        src = batch["src_img"].to(device, non_blocking=True)
        tgt = batch["tgt_img"].to(device, non_blocking=True)
        ids = batch["input_ids"].to(device, non_blocking=True)
        mask = batch["attention_mask"].to(device, non_blocking=True)
        tgts = batch["target"].to(device, non_blocking=True)
        op_ids = batch["op_type_id"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", enabled=(USE_AMP and device.type == "cuda")):
            logits, evidence = model(src, tgt, ids, mask)
            loss_tax = criterion_tax(logits, tgts)
            loss_op = criterion_op(evidence["op_logits"], op_ids)
            loss = loss_tax + AUX_OP_LOSS_WEIGHT * loss_op

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        running_loss += float(loss.item()) * tgts.shape[0]
        n += int(tgts.shape[0])

    train_loss = running_loss / max(n, 1)

    # model selection on taxonomy-positive val benchmark for direct comparability
    val_loss, y_true_v, y_prob_v, _, _, _, _ = run_eval(
        model,
        val_loader_benchmark,
        criterion_tax,
        criterion_op,
        device,
        use_amp=USE_AMP,
    )
    thresholds_v = tune_thresholds(y_true_v, y_prob_v)
    y_pred_v = (y_prob_v >= np.array(thresholds_v).reshape(1, -1)).astype(int)
    metrics_v = compute_multilabel_metrics(y_true_v, y_prob_v, y_pred_v)

    score = metrics_v["macro_f1_supported"]
    print(
        f"Epoch {epoch:02d} | train_loss={train_loss:.4f} "
        f"val_loss={val_loss:.4f} | micro_f1={metrics_v['micro_f1']:.4f} "
        f"macro_f1_sup={score:.4f} | mAP_micro={metrics_v['mAP_micro']:.4f}"
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss_benchmark": val_loss,
        **{f"val_benchmark_{k}": v for k, v in metrics_v.items()},
    })

    if score > best_score:
        best_score = score
        best_state = copy.deepcopy(model.state_dict())
        best_thresholds = thresholds_v
        no_improve = 0
        torch.save(best_state, WORK_DIR / "best_model.pt")
        save_json(WORK_DIR / "best_thresholds.json", {cls: float(t) for cls, t in zip(ERROR_TYPES, best_thresholds)})
        print(f"  ✓ saved new best (macro_f1_supported={score:.4f})")
    else:
        no_improve += 1
        if no_improve >= PATIENCE and not SMOKE_TEST:
            print(f"  Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
            break

save_json(WORK_DIR / "train_history.json", {"history": history})
print(f"\nTraining done. Best val macro_f1_supported = {best_score:.4f}")

Epoch 01 | train_loss=0.5668 val_loss=0.9422 | micro_f1=0.5351 macro_f1_sup=0.4315 | mAP_micro=0.4807
  ✓ saved new best (macro_f1_supported=0.4315)
Epoch 02 | train_loss=0.2480 val_loss=1.5272 | micro_f1=0.5617 macro_f1_sup=0.4481 | mAP_micro=0.4944
  ✓ saved new best (macro_f1_supported=0.4481)
Epoch 03 | train_loss=0.1445 val_loss=2.4545 | micro_f1=0.6222 macro_f1_sup=0.4591 | mAP_micro=0.5757
  ✓ saved new best (macro_f1_supported=0.4591)
Epoch 04 | train_loss=0.0992 val_loss=2.6371 | micro_f1=0.6617 macro_f1_sup=0.4865 | mAP_micro=0.5770
  ✓ saved new best (macro_f1_supported=0.4865)
Epoch 05 | train_loss=0.0768 val_loss=3.5434 | micro_f1=0.6312 macro_f1_sup=0.4656 | mAP_micro=0.5616
Epoch 06 | train_loss=0.0665 val_loss=3.6914 | micro_f1=0.6450 macro_f1_sup=0.4560 | mAP_micro=0.6032
Epoch 07 | train_loss=0.0544 val_loss=4.2465 | micro_f1=0.6320 macro_f1_sup=0.4477 | mAP_micro=0.5608
  Early stopping at epoch 7 (no improvement for 3 epochs)

Training done. Best val macro_f1_suppor

In [12]:
# Cell 12 — final evaluation and artifact export

if best_state is not None:
    model.load_state_dict(best_state)
else:
    model.load_state_dict(torch.load(WORK_DIR / "best_model.pt", map_location=device))

# Benchmark eval (taxonomy-positive; this is the primary comparable benchmark)
test_bench_loss, y_true_tb, y_prob_tb, y_logit_tb, langs_tb, ids_tb, evidence_tb = run_eval(
    model,
    test_loader_benchmark,
    criterion_tax,
    criterion_op,
    device,
    use_amp=USE_AMP,
)
y_pred_tb = (y_prob_tb >= np.array(best_thresholds).reshape(1, -1)).astype(int)
metrics_tb = compute_multilabel_metrics(y_true_tb, y_prob_tb, y_pred_tb)

print("\n── Primary benchmark: taxonomy-positive held-out test ───────────────")
for k, v in metrics_tb.items():
    print(f"  {k:<24}: {v:.4f}")

per_class_tb = compute_per_class_metrics(y_true_tb, y_prob_tb, y_pred_tb)
per_class_tb.to_csv(WORK_DIR / "per_class_test_benchmark.csv", index=False)

# Optional full test eval for calibration visibility
test_full_loss, y_true_tf, y_prob_tf, y_logit_tf, langs_tf, ids_tf, evidence_tf = run_eval(
    model,
    test_loader_full,
    criterion_tax,
    criterion_op,
    device,
    use_amp=USE_AMP,
)
y_pred_tf = (y_prob_tf >= np.array(best_thresholds).reshape(1, -1)).astype(int)
metrics_tf = compute_multilabel_metrics(y_true_tf, y_prob_tf, y_pred_tf)

print("\n── Secondary eval: full held-out test rows ──────────────────────────")
for k, v in metrics_tf.items():
    print(f"  {k:<24}: {v:.4f}")

save_json(WORK_DIR / "metrics.json", {
    "primary_eval_mode": PRIMARY_EVAL_MODE,
    "val_best_macro_f1_supported": float(best_score),
    "test_benchmark": metrics_tb,
    "test_full": metrics_tf,
    "best_thresholds": {cls: float(t) for cls, t in zip(ERROR_TYPES, best_thresholds)},
    "config": {
        "vision_model_id": VISION_MODEL_ID,
        "text_model_id": TEXT_MODEL_ID,
        "hidden_dim": HIDDEN_DIM,
        "train_neg_ratio": TRAIN_NEG_RATIO,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "aux_op_loss_weight": AUX_OP_LOSS_WEIGHT,
        "unfreeze_text_layers": UNFREEZE_TEXT_LAYERS,
        "unfreeze_vision_layers": UNFREEZE_VISION_LAYERS,
    },
})

save_json(WORK_DIR / "label_map.json", {str(i): c for i, c in enumerate(ERROR_TYPES)})

# Save row-level predictions on benchmark set
benchmark_pred_rows = []
for i in range(len(ids_tb)):
    benchmark_pred_rows.append({
        "id": ids_tb[i],
        "lang": langs_tb[i],
        "gold_labels": [ERROR_TYPES[j] for j, v in enumerate(y_true_tb[i]) if int(v) == 1],
        "pred_labels": [ERROR_TYPES[j] for j, v in enumerate(y_pred_tb[i]) if int(v) == 1],
        "probs": {ERROR_TYPES[j]: float(y_prob_tb[i, j]) for j in range(NUM_CLASSES)},
    })

with open(WORK_DIR / "predictions_test_benchmark.jsonl", "w", encoding="utf-8") as f:
    for row in benchmark_pred_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

evidence_tb.to_csv(WORK_DIR / "evidence_test_benchmark.csv", index=False)

print("\nSaved artifacts:")
for p in [
    WORK_DIR / "best_model.pt",
    WORK_DIR / "best_thresholds.json",
    WORK_DIR / "metrics.json",
    WORK_DIR / "per_class_test_benchmark.csv",
    WORK_DIR / "predictions_test_benchmark.jsonl",
    WORK_DIR / "evidence_test_benchmark.csv",
]:
    print(" ", p)


── Primary benchmark: taxonomy-positive held-out test ───────────────
  micro_f1                : 0.6557
  macro_f1                : 0.2799
  macro_f1_supported      : 0.3421
  mAP_micro               : 0.5516
  mAP_macro_supported     : 0.3301
                       class        f1  precision    recall  support
0               Wrong Object  0.230769   0.272727  0.200000       15
1             Missing Object  0.134615   0.170732  0.111111       63
2               Extra Object  0.000000   0.000000  0.000000        0
3            Wrong Attribute  0.350598   0.478261  0.276730      159
4              Spatial Error  0.216867   0.391304  0.150000       60
5             Style Mismatch  0.431461   0.350365  0.561404      171
6               Over-editing  0.296296   0.285714  0.307692       39
7              Under-editing  0.890896   0.826031  0.966817      663
8   Artifact / Quality Issue  0.142857   0.142857  0.142857       21
9           Ambiguous Prompt  0.000000   0.000000  0.000000     

In [13]:
# Cell 13 — optional evidence inspection on a few benchmark rows

def evidence_to_reason(row: pd.Series, pred_labels: List[str]) -> str:
    parts = []
    parts.append(f"pred_op={row['pred_op_type']}")
    parts.append(f"src_presence={row['src_presence']:.2f}")
    parts.append(f"tgt_presence={row['tgt_presence']:.2f}")
    parts.append(f"corr_max={row['correspondence_max']:.2f}")
    parts.append(f"local_change={row['local_change_score']:.2f}")
    parts.append(f"outside_change={row['outside_change_score']:.2f}")
    if pred_labels:
        parts.append("labels=" + ", ".join(pred_labels))
    else:
        parts.append("labels=[]")
    return " | ".join(parts)

evidence_view = evidence_tb.copy()
pred_lookup = {(r["id"], r["lang"]): r["pred_labels"] for r in benchmark_pred_rows}
evidence_view["pred_labels"] = evidence_view.apply(
    lambda x: pred_lookup.get((x["id"], x["lang"]), []), axis=1
)
evidence_view["reason"] = evidence_view.apply(
    lambda x: evidence_to_reason(x, x["pred_labels"]), axis=1
)

display(evidence_view.head(12))
evidence_view.head(50).to_csv(WORK_DIR / "evidence_preview.csv", index=False)
print("Saved:", WORK_DIR / "evidence_preview.csv")

,id,lang,src_presence,tgt_presence,correspondence_max,correspondence_mean,local_change_score,global_change_score,outside_change_score,pred_op_type,pred_labels,reason
0,pref_02333,en,0.000000e+00,1.000000,0.907715,0.822754,0.194214,0.017639,0.143677,recolor,[Under-editing],pred_op=recolor | src_presence=0.00 | tgt_pres...
1,pref_02333,hi,1.192093e-07,1.000000,0.907715,0.822754,0.194214,0.017639,0.143677,recolor,[Under-editing],pred_op=recolor | src_presence=0.00 | tgt_pres...
2,pref_02333,bn,7.489014e-02,0.997070,0.907715,0.822754,0.194336,0.017639,0.143677,remove,[],pred_op=remove | src_presence=0.07 | tgt_prese...
3,pref_02339,en,3.576279e-07,0.000182,0.874512,0.797852,0.223511,0.804199,0.173584,other,[Under-editing],pred_op=other | src_presence=0.00 | tgt_presen...
4,pref_02339,hi,1.000000e+00,1.000000,0.874512,0.797852,0.223511,0.804199,0.173584,add,[Under-editing],pred_op=add | src_presence=1.00 | tgt_presence...
5,pref_02339,bn,1.000000e+00,1.000000,0.874512,0.797852,0.223511,0.804199,0.173584,add,[Under-editing],pred_op=add | src_presence=1.00 | tgt_presence...
6,pref_02344,en,2.861023e-06,1.000000,0.894531,0.793457,0.217041,0.052032,0.166016,recolor,[Under-editing],pred_op=recolor | src_presence=0.00 | tgt_pres...
7,pref_02344,hi,5.692244e-05,1.000000,0.894531,0.792969,0.217041,0.052032,0.166016,recolor,[Under-editing],pred_op=recolor | src_presence=0.00 | tgt_pres...
8,pref_02344,bn,5.859375e-01,0.994629,0.894043,0.793457,0.216797,0.052032,0.166016,background,[Under-editing],pred_op=background | src_presence=0.59 | tgt_p...
9,pref_02345,en,3.647804e-05,1.000000,0.943359,0.815918,0.280029,0.303467,0.229370,remove,"[Wrong Attribute, Under-editing]",pred_op=remove | src_presence=0.00 | tgt_prese...


Saved: /kaggle/working/evidence_grounded_taxonomy_eval/evidence_preview.csv


In [14]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, average_precision_score

# ── robust resolver from existing notebook state ──────────────────────────────
def _pick_first(name_list):
    for n in name_list:
        if n in globals() and globals()[n] is not None:
            return globals()[n], n
    return None, None

def _find_loader_by_keywords(required_keywords, exclude_keywords=None):
    exclude_keywords = exclude_keywords or []
    candidates = []
    for name, obj in globals().items():
        if isinstance(obj, DataLoader):
            lname = name.lower()
            if all(k in lname for k in required_keywords) and not any(x in lname for x in exclude_keywords):
                candidates.append((name, obj))
    if candidates:
        # shortest name first, usually the intended canonical variable
        candidates = sorted(candidates, key=lambda x: len(x[0]))
        return candidates[0][1], candidates[0][0]
    return None, None

model, model_name = _pick_first(["model"])
device, device_name = _pick_first(["device"])

# first try canonical names
train_loader, train_loader_name = _pick_first([
    "train_loader", "train_dl", "loader_train", "dl_train"
])
val_loader, val_loader_name = _pick_first([
    "val_loader", "valid_loader", "validation_loader", "val_dl", "loader_val", "dl_val"
])
test_loader_benchmark, test_loader_benchmark_name = _pick_first([
    "test_loader_benchmark", "benchmark_loader", "test_benchmark_loader",
    "test_loader_taxonomy_positive", "benchmark_test_loader"
])
test_loader_full, test_loader_full_name = _pick_first([
    "test_loader_full", "full_test_loader", "test_full_loader",
    "test_loader_all", "heldout_full_loader"
])

# fallback: search by keywords
if train_loader is None:
    train_loader, train_loader_name = _find_loader_by_keywords(["train"])
if val_loader is None:
    val_loader, val_loader_name = _find_loader_by_keywords(["val"])
if val_loader is None:
    val_loader, val_loader_name = _find_loader_by_keywords(["valid"])
if test_loader_benchmark is None:
    test_loader_benchmark, test_loader_benchmark_name = _find_loader_by_keywords(["benchmark"])
if test_loader_benchmark is None:
    test_loader_benchmark, test_loader_benchmark_name = _find_loader_by_keywords(["test", "bench"])
if test_loader_benchmark is None:
    test_loader_benchmark, test_loader_benchmark_name = _find_loader_by_keywords(["test", "taxonomy"])
if test_loader_full is None:
    test_loader_full, test_loader_full_name = _find_loader_by_keywords(["test", "full"])
if test_loader_full is None:
    test_loader_full, test_loader_full_name = _find_loader_by_keywords(["full", "test"])

# last fallback: rebuild loaders from dataset objects if loaders were not kept
batch_size_guess = globals().get("BATCH_SIZE", 8)
num_workers_guess = globals().get("NUM_WORKERS", 0)

if train_loader is None:
    train_ds, train_ds_name = _pick_first(["train_ds", "dataset_train", "train_dataset"])
    if train_ds is not None:
        train_loader = DataLoader(train_ds, batch_size=batch_size_guess, shuffle=True, num_workers=num_workers_guess)
        train_loader_name = f"rebuilt_from_{train_ds_name}"

if val_loader is None:
    val_ds, val_ds_name = _pick_first(["val_ds", "valid_ds", "dataset_val", "val_dataset"])
    if val_ds is not None:
        val_loader = DataLoader(val_ds, batch_size=batch_size_guess, shuffle=False, num_workers=num_workers_guess)
        val_loader_name = f"rebuilt_from_{val_ds_name}"

if test_loader_benchmark is None:
    bench_ds, bench_ds_name = _pick_first([
        "test_benchmark_ds", "benchmark_ds", "benchmark_test_ds", "test_ds_benchmark"
    ])
    if bench_ds is not None:
        test_loader_benchmark = DataLoader(bench_ds, batch_size=batch_size_guess, shuffle=False, num_workers=num_workers_guess)
        test_loader_benchmark_name = f"rebuilt_from_{bench_ds_name}"

if test_loader_full is None:
    full_ds, full_ds_name = _pick_first([
        "test_full_ds", "full_test_ds", "test_ds_full", "heldout_full_ds"
    ])
    if full_ds is not None:
        test_loader_full = DataLoader(full_ds, batch_size=batch_size_guess, shuffle=False, num_workers=num_workers_guess)
        test_loader_full_name = f"rebuilt_from_{full_ds_name}"

if model is None:
    raise RuntimeError("Could not find `model` in notebook globals.")
if device is None:
    raise RuntimeError("Could not find `device` in notebook globals.")
if train_loader is None or val_loader is None:
    print("Available DataLoader globals:")
    for name, obj in globals().items():
        if isinstance(obj, DataLoader):
            print("  ", name)
    raise RuntimeError("Could not resolve `train_loader` / `val_loader`. See printed DataLoader globals above.")
if test_loader_benchmark is None:
    raise RuntimeError("Could not find taxonomy-positive benchmark test loader in notebook globals.")
if test_loader_full is None:
    raise RuntimeError("Could not find full held-out test loader in notebook globals.")

print("Using:")
print("  model                =", model_name)
print("  device               =", device_name)
print("  train_loader         =", train_loader_name)
print("  val_loader           =", val_loader_name)
print("  test_loader_benchmark=", test_loader_benchmark_name)
print("  test_loader_full     =", test_loader_full_name)

Using:
  model                = model
  device               = device
  train_loader         = train_loader
  val_loader           = val_loader_full
  test_loader_benchmark= test_loader_benchmark
  test_loader_full     = test_loader_full


In [15]:
# Prefer taxonomy-focused validation loader for model selection if available
if "val_loader_benchmark" in globals() and val_loader_benchmark is not None:
    val_loader = val_loader_benchmark
    val_loader_name = "val_loader_benchmark"
elif "benchmark_val_loader" in globals() and benchmark_val_loader is not None:
    val_loader = benchmark_val_loader
    val_loader_name = "benchmark_val_loader"
elif "val_loader_taxonomy_positive" in globals() and val_loader_taxonomy_positive is not None:
    val_loader = val_loader_taxonomy_positive
    val_loader_name = "val_loader_taxonomy_positive"

print("Final v2 selection loaders:")
print("  train_loader         =", train_loader_name)
print("  val_loader           =", val_loader_name)
print("  test_loader_benchmark=", test_loader_benchmark_name)
print("  test_loader_full     =", test_loader_full_name)

Final v2 selection loaders:
  train_loader         = train_loader
  val_loader           = val_loader_benchmark
  test_loader_benchmark= test_loader_benchmark
  test_loader_full     = test_loader_full


In [16]:
# Cell 12 — v2 fine-tuning from saved v1 best (same protocol, weighted taxonomy loss)

import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, average_precision_score

# ── resolve required objects from existing notebook state ─────────────────────
def _pick_first(name_list):
    for n in name_list:
        if n in globals() and globals()[n] is not None:
            return globals()[n], n
    return None, None

model, model_name = _pick_first(["model"])
device, device_name = _pick_first(["device"])
train_loader, train_loader_name = _pick_first(["train_loader"])
val_loader, val_loader_name = _pick_first(["val_loader"])
test_loader_benchmark, test_loader_benchmark_name = _pick_first([
    "test_loader_benchmark", "benchmark_loader", "test_benchmark_loader"
])
test_loader_full, test_loader_full_name = _pick_first([
    "test_loader_full", "full_test_loader", "test_full_loader"
])

if model is None:
    raise RuntimeError("Could not find `model` in notebook globals.")
if device is None:
    raise RuntimeError("Could not find `device` in notebook globals.")
if train_loader is None or val_loader is None:
    raise RuntimeError("Could not find `train_loader` / `val_loader` in notebook globals.")
if test_loader_benchmark is None:
    raise RuntimeError("Could not find taxonomy-positive benchmark test loader in notebook globals.")
if test_loader_full is None:
    raise RuntimeError("Could not find full held-out test loader in notebook globals.")

print("Using:")
print("  model                =", model_name)
print("  device               =", device_name)
print("  train_loader         =", train_loader_name)
print("  val_loader           =", val_loader_name)
print("  test_loader_benchmark=", test_loader_benchmark_name)
print("  test_loader_full     =", test_loader_full_name)

# ── output dirs and checkpoint paths ──────────────────────────────────────────
V1_DIR = Path("/kaggle/working/evidence_grounded_taxonomy_eval")
V2_DIR = Path("/kaggle/working/evidence_grounded_taxonomy_eval_v2")
V2_DIR.mkdir(parents=True, exist_ok=True)

V1_CKPT = V1_DIR / "best_model.pt"
if not V1_CKPT.exists():
    raise FileNotFoundError(f"Missing v1 checkpoint: {V1_CKPT}")

print("Loading v1 best checkpoint from:", V1_CKPT)
state = torch.load(V1_CKPT, map_location=device)
model.load_state_dict(state)
model.to(device)

# ── helpers ───────────────────────────────────────────────────────────────────
def _batch_to_device(batch, device):
    src = batch["src_img"].to(device, non_blocking=True)
    tgt = batch["tgt_img"].to(device, non_blocking=True)
    ids = batch["input_ids"].to(device, non_blocking=True)
    mask = batch["attention_mask"].to(device, non_blocking=True)
    tgts = batch["target"].to(device, non_blocking=True).float()
    return src, tgt, ids, mask, tgts

def _collect_train_targets(loader):
    ys = []
    for batch in loader:
        ys.append(batch["target"].cpu().numpy())
    return np.concatenate(ys, axis=0)

def _safe_macro_supported_f1(y_true, y_pred):
    per = f1_score(y_true, y_pred, average=None, zero_division=0)
    support_mask = y_true.sum(axis=0) > 0
    return float(np.mean(per[support_mask])) if support_mask.any() else 0.0

def _tune_thresholds(y_true, y_prob):
    # per-class threshold tuning on validation set
    grid = np.arange(0.10, 0.91, 0.05)
    n_classes = y_true.shape[1]
    best = []
    for c in range(n_classes):
        yt = y_true[:, c]
        yp = y_prob[:, c]
        if yt.sum() == 0:
            best.append(0.50)
            continue
        best_t = 0.50
        best_f = -1.0
        for t in grid:
            pred = (yp >= t).astype(int)
            f = f1_score(yt, pred, zero_division=0)
            if f > best_f:
                best_f = f
                best_t = float(t)
        best.append(best_t)
    return best

def _compute_metrics(y_true, y_prob, y_pred):
    out = {}
    out["micro_f1"] = float(f1_score(y_true, y_pred, average="micro", zero_division=0))
    out["macro_f1"] = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    out["macro_f1_supported"] = _safe_macro_supported_f1(y_true, y_pred)

    try:
        out["mAP_micro"] = float(average_precision_score(y_true, y_prob, average="micro"))
    except Exception:
        out["mAP_micro"] = 0.0

    ap_supported = []
    for i in range(y_true.shape[1]):
        if y_true[:, i].sum() > 0:
            try:
                ap_supported.append(float(average_precision_score(y_true[:, i], y_prob[:, i])))
            except Exception:
                pass
    out["mAP_macro_supported"] = float(np.mean(ap_supported)) if ap_supported else 0.0
    return out

def _per_class_df(y_true, y_pred, class_names):
    rows = []
    per_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
    for i, cls in enumerate(class_names):
        yt = y_true[:, i]
        yp = y_pred[:, i]
        tp = int(((yt == 1) & (yp == 1)).sum())
        pp = int((yp == 1).sum())
        ap = int((yt == 1).sum())
        prec = tp / pp if pp > 0 else 0.0
        rec = tp / ap if ap > 0 else 0.0
        rows.append({
            "class": cls,
            "f1": float(per_f1[i]),
            "precision": float(prec),
            "recall": float(rec),
            "support": int(ap),
        })
    return pd.DataFrame(rows)

def _eval_loader(model, loader, criterion_tax, device, use_amp=True):
    model.eval()
    total_loss = 0.0
    n = 0
    y_true, y_prob = [], []

    with torch.no_grad():
        for batch in loader:
            src, tgt, ids, mask, tgts = _batch_to_device(batch, device)

            with torch.autocast(device_type="cuda", enabled=(use_amp and device.type == "cuda")):
                logits, evidence = model(src, tgt, ids, mask)
                loss_tax = criterion_tax(logits, tgts)

            probs = torch.sigmoid(logits).detach().cpu().numpy()
            y_prob.append(probs)
            y_true.append(tgts.detach().cpu().numpy())
            total_loss += float(loss_tax.item()) * tgts.shape[0]
            n += int(tgts.shape[0])

    y_true = np.concatenate(y_true, axis=0)
    y_prob = np.concatenate(y_prob, axis=0)
    return total_loss / max(n, 1), y_true, y_prob

# ── v2 tuning changes ─────────────────────────────────────────────────────────
# 1) warm-start from best v1 checkpoint
# 2) class-weighted taxonomy BCE to help minority classes
# 3) lower LR + stronger weight decay
# 4) taxonomy-only fine-tuning (aux heads stay in graph but no aux loss)
# 5) re-tune thresholds on val every epoch
# 6) early stop on val macro_f1_supported

V2_EPOCHS = 5
V2_PATIENCE = 3
V2_LR = 1e-5
V2_WEIGHT_DECAY = 5e-4
V2_USE_AMP = True
V2_GRAD_CLIP = 1.0

train_targets = _collect_train_targets(train_loader)
pos = train_targets.sum(axis=0)
neg = train_targets.shape[0] - pos
pos_weight = (neg / np.clip(pos, 1, None)).astype(np.float32)
pos_weight = np.clip(pos_weight, 1.0, 8.0)  # keep stable
pos_weight_t = torch.tensor(pos_weight, dtype=torch.float32, device=device)

print("v2 pos_weight:")
for i, w in enumerate(pos_weight.tolist()):
    print(f"  {ERROR_TYPES[i]:<26} {w:.3f}")

criterion_tax_v2 = nn.BCEWithLogitsLoss(pos_weight=pos_weight_t)

optimizer_v2 = torch.optim.AdamW(
    model.parameters(),
    lr=V2_LR,
    weight_decay=V2_WEIGHT_DECAY,
)

best_score = -1.0
best_thresholds = [0.5] * len(ERROR_TYPES)
no_improve = 0
history_v2 = []

for epoch in range(1, V2_EPOCHS + 1):
    model.train()
    run_loss = 0.0
    n = 0

    for batch in train_loader:
        src, tgt, ids, mask, tgts = _batch_to_device(batch, device)

        optimizer_v2.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", enabled=(V2_USE_AMP and device.type == "cuda")):
            logits, evidence = model(src, tgt, ids, mask)
            loss = criterion_tax_v2(logits, tgts)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), V2_GRAD_CLIP)
        optimizer_v2.step()

        run_loss += float(loss.item()) * tgts.shape[0]
        n += int(tgts.shape[0])

    val_loss, y_true_v, y_prob_v = _eval_loader(model, val_loader, criterion_tax_v2, device, use_amp=V2_USE_AMP)
    thresholds_v = _tune_thresholds(y_true_v, y_prob_v)
    y_pred_v = (y_prob_v >= np.array(thresholds_v).reshape(1, -1)).astype(int)
    metrics_v = _compute_metrics(y_true_v, y_prob_v, y_pred_v)

    score = metrics_v["macro_f1_supported"]
    print(
        f"v2 Epoch {epoch:02d} | train_loss={run_loss/max(n,1):.4f} "
        f"val_loss={val_loss:.4f} | micro_f1={metrics_v['micro_f1']:.4f} "
        f"macro_f1_sup={metrics_v['macro_f1_supported']:.4f} | mAP_micro={metrics_v['mAP_micro']:.4f}"
    )

    history_v2.append({
        "epoch": epoch,
        "train_loss": run_loss / max(n, 1),
        "val_loss": val_loss,
        **{f"val_{k}": v for k, v in metrics_v.items()},
    })

    if score > best_score:
        best_score = score
        best_thresholds = thresholds_v
        torch.save(model.state_dict(), V2_DIR / "best_model.pt")
        print(f"  ✓ saved new v2 best (macro_f1_supported={score:.4f})")
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= V2_PATIENCE:
            print(f"  Early stopping v2 at epoch {epoch} (no improvement for {V2_PATIENCE} epochs)")
            break

print(f"\nV2 fine-tuning done. Best val macro_f1_supported = {best_score:.4f}")

# ── load best v2 and evaluate on benchmark + full held-out test ──────────────
best_state = torch.load(V2_DIR / "best_model.pt", map_location=device)
model.load_state_dict(best_state)
model.to(device)
model.eval()

# primary benchmark: taxonomy-positive held-out test
test_bench_loss, y_true_b, y_prob_b = _eval_loader(model, test_loader_benchmark, criterion_tax_v2, device, use_amp=V2_USE_AMP)
y_pred_b = (y_prob_b >= np.array(best_thresholds).reshape(1, -1)).astype(int)
metrics_b = _compute_metrics(y_true_b, y_prob_b, y_pred_b)

print("\n── V2 Primary benchmark: taxonomy-positive held-out test ─────────────")
for k, v in metrics_b.items():
    print(f"  {k:<24}: {v:.4f}")

per_class_b = _per_class_df(y_true_b, y_pred_b, ERROR_TYPES)
display(per_class_b)

# secondary: full held-out test
test_full_loss, y_true_f, y_prob_f = _eval_loader(model, test_loader_full, criterion_tax_v2, device, use_amp=V2_USE_AMP)
y_pred_f = (y_prob_f >= np.array(best_thresholds).reshape(1, -1)).astype(int)
metrics_f = _compute_metrics(y_true_f, y_prob_f, y_pred_f)

print("\n── V2 Secondary eval: full held-out test rows ────────────────────────")
for k, v in metrics_f.items():
    print(f"  {k:<24}: {v:.4f}")

# save artifacts
(V2_DIR / "train_history_v2.json").write_text(json.dumps(history_v2, indent=2), encoding="utf-8")
(V2_DIR / "best_thresholds.json").write_text(json.dumps(best_thresholds, indent=2), encoding="utf-8")

metrics_out = {
    "v2_best_val_macro_f1_supported": best_score,
    "v2_benchmark_metrics": metrics_b,
    "v2_full_test_metrics": metrics_f,
    "best_thresholds": best_thresholds,
    "pos_weight": pos_weight.tolist(),
}
(V2_DIR / "metrics.json").write_text(json.dumps(metrics_out, indent=2), encoding="utf-8")

per_class_b.to_csv(V2_DIR / "per_class_test_benchmark.csv", index=False)

# save predictions
with open(V2_DIR / "predictions_test_benchmark.jsonl", "w", encoding="utf-8") as f:
    for i in range(y_true_b.shape[0]):
        rec = {
            "gold_vec": y_true_b[i].astype(int).tolist(),
            "pred_vec": y_pred_b[i].astype(int).tolist(),
            "prob": [float(x) for x in y_prob_b[i].tolist()],
        }
        f.write(json.dumps(rec) + "\n")

print("\nSaved v2 artifacts:")
print(" ", V2_DIR / "best_model.pt")
print(" ", V2_DIR / "best_thresholds.json")
print(" ", V2_DIR / "metrics.json")
print(" ", V2_DIR / "per_class_test_benchmark.csv")
print(" ", V2_DIR / "predictions_test_benchmark.jsonl")

Using:
  model                = model
  device               = device
  train_loader         = train_loader
  val_loader           = val_loader
  test_loader_benchmark= test_loader_benchmark
  test_loader_full     = test_loader_full
Loading v1 best checkpoint from: /kaggle/working/evidence_grounded_taxonomy_eval/best_model.pt
v2 pos_weight:
  Wrong Object               8.000
  Missing Object             8.000
  Extra Object               8.000
  Wrong Attribute            6.236
  Spatial Error              8.000
  Style Mismatch             5.618
  Over-editing               8.000
  Under-editing              1.000
  Artifact / Quality Issue   8.000
  Ambiguous Prompt           8.000
  Failed Removal             8.000
v2 Epoch 01 | train_loss=0.0660 val_loss=2.5439 | micro_f1=0.6707 macro_f1_sup=0.4917 | mAP_micro=0.6205
  ✓ saved new v2 best (macro_f1_supported=0.4917)
v2 Epoch 02 | train_loss=0.0373 val_loss=3.0222 | micro_f1=0.6677 macro_f1_sup=0.4658 | mAP_micro=0.6325
v2 Epoch 03 

,class,f1,precision,recall,support
0,Wrong Object,0.250000,0.333333,0.200000,15
1,Missing Object,0.126316,0.187500,0.095238,63
2,Extra Object,0.000000,0.000000,0.000000,0
3,Wrong Attribute,0.418033,0.600000,0.320755,159
4,Spatial Error,0.222222,0.428571,0.150000,60
5,Style Mismatch,0.435233,0.390698,0.491228,171
6,Over-editing,0.333333,0.600000,0.230769,39
7,Under-editing,0.873016,0.836791,0.912519,663
8,Artifact / Quality Issue,0.193548,0.300000,0.142857,21
9,Ambiguous Prompt,0.000000,0.000000,0.000000,0



── V2 Secondary eval: full held-out test rows ────────────────────────
  micro_f1                : 0.4659
  macro_f1                : 0.2182
  macro_f1_supported      : 0.2666
  mAP_micro               : 0.4056
  mAP_macro_supported     : 0.2369

Saved v2 artifacts:
  /kaggle/working/evidence_grounded_taxonomy_eval_v2/best_model.pt
  /kaggle/working/evidence_grounded_taxonomy_eval_v2/best_thresholds.json
  /kaggle/working/evidence_grounded_taxonomy_eval_v2/metrics.json
  /kaggle/working/evidence_grounded_taxonomy_eval_v2/per_class_test_benchmark.csv
  /kaggle/working/evidence_grounded_taxonomy_eval_v2/predictions_test_benchmark.jsonl


In [17]:
# Cell 13 — v3 fine-tuning from v2 best (balance F1 + calibration)

import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, average_precision_score

# ── resolve required objects from current notebook state ──────────────────────
required = ["model", "device", "train_loader", "val_loader", "test_loader_benchmark", "test_loader_full"]
missing = [x for x in required if x not in globals() or globals()[x] is None]
if missing:
    raise RuntimeError(f"Missing required globals for v3: {missing}")

# Optional: use taxonomy-focused val loader if present
if "val_loader_benchmark" in globals() and val_loader_benchmark is not None:
    val_loader = val_loader_benchmark
    print("Using val_loader_benchmark for v3 selection.")
elif "benchmark_val_loader" in globals() and benchmark_val_loader is not None:
    val_loader = benchmark_val_loader
    print("Using benchmark_val_loader for v3 selection.")
else:
    print("Using current val_loader for v3 selection.")

# ── output dirs and checkpoints ───────────────────────────────────────────────
V1_DIR = Path("/kaggle/working/evidence_grounded_taxonomy_eval")
V2_DIR = Path("/kaggle/working/evidence_grounded_taxonomy_eval_v2")
V3_DIR = Path("/kaggle/working/evidence_grounded_taxonomy_eval_v3")
V3_DIR.mkdir(parents=True, exist_ok=True)

if (V2_DIR / "best_model.pt").exists():
    START_CKPT = V2_DIR / "best_model.pt"
    print("Warm-starting from v2:", START_CKPT)
elif (V1_DIR / "best_model.pt").exists():
    START_CKPT = V1_DIR / "best_model.pt"
    print("Warm-starting from v1:", START_CKPT)
else:
    raise FileNotFoundError("Could not find v1/v2 checkpoint to warm-start v3.")

state = torch.load(START_CKPT, map_location=device)
model.load_state_dict(state)
model.to(device)
model.train()

# ── helper functions ──────────────────────────────────────────────────────────
def _batch_to_device(batch, device):
    src = batch["src_img"].to(device, non_blocking=True)
    tgt = batch["tgt_img"].to(device, non_blocking=True)
    ids = batch["input_ids"].to(device, non_blocking=True)
    mask = batch["attention_mask"].to(device, non_blocking=True)
    tgts = batch["target"].to(device, non_blocking=True).float()
    return src, tgt, ids, mask, tgts

def _get_op_ids_from_batch(batch, device):
    for key in ["op_id", "op_ids", "operation_id", "operation_ids"]:
        if key in batch:
            return batch[key].to(device, non_blocking=True).long(), key
    return None, None

def _collect_train_targets(loader):
    ys = []
    for batch in loader:
        ys.append(batch["target"].cpu().numpy())
    return np.concatenate(ys, axis=0)

def _safe_macro_supported_f1(y_true, y_pred):
    per = f1_score(y_true, y_pred, average=None, zero_division=0)
    support_mask = y_true.sum(axis=0) > 0
    return float(np.mean(per[support_mask])) if support_mask.any() else 0.0

def _tune_thresholds(y_true, y_prob):
    grid = np.arange(0.10, 0.91, 0.05)
    n_classes = y_true.shape[1]
    best = []
    for c in range(n_classes):
        yt = y_true[:, c]
        yp = y_prob[:, c]
        if yt.sum() == 0:
            best.append(0.50)
            continue
        best_t = 0.50
        best_f = -1.0
        for t in grid:
            pred = (yp >= t).astype(int)
            f = f1_score(yt, pred, zero_division=0)
            if f > best_f:
                best_f = f
                best_t = float(t)
        best.append(best_t)
    return best

def _compute_metrics(y_true, y_prob, y_pred):
    out = {}
    out["micro_f1"] = float(f1_score(y_true, y_pred, average="micro", zero_division=0))
    out["macro_f1"] = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    out["macro_f1_supported"] = _safe_macro_supported_f1(y_true, y_pred)
    try:
        out["mAP_micro"] = float(average_precision_score(y_true, y_prob, average="micro"))
    except Exception:
        out["mAP_micro"] = 0.0

    ap_supported = []
    for i in range(y_true.shape[1]):
        if y_true[:, i].sum() > 0:
            try:
                ap_supported.append(float(average_precision_score(y_true[:, i], y_prob[:, i])))
            except Exception:
                pass
    out["mAP_macro_supported"] = float(np.mean(ap_supported)) if ap_supported else 0.0
    return out

def _per_class_df(y_true, y_pred, class_names):
    rows = []
    per_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
    for i, cls in enumerate(class_names):
        yt = y_true[:, i]
        yp = y_pred[:, i]
        tp = int(((yt == 1) & (yp == 1)).sum())
        pp = int((yp == 1).sum())
        ap = int((yt == 1).sum())
        prec = tp / pp if pp > 0 else 0.0
        rec = tp / ap if ap > 0 else 0.0
        rows.append({
            "class": cls,
            "f1": float(per_f1[i]),
            "precision": float(prec),
            "recall": float(rec),
            "support": int(ap),
        })
    return pd.DataFrame(rows)

def _eval_loader_v3(model, loader, criterion_plain, criterion_weighted, device, use_amp=True):
    model.eval()
    total_loss = 0.0
    n = 0
    y_true, y_prob = [], []

    with torch.no_grad():
        for batch in loader:
            src, tgt, ids, mask, tgts = _batch_to_device(batch, device)

            with torch.autocast(device_type="cuda", enabled=(use_amp and device.type == "cuda")):
                logits, evidence = model(src, tgt, ids, mask)
                loss_plain = criterion_plain(logits, tgts)
                loss_weighted = criterion_weighted(logits, tgts)
                loss_tax = V3_ALPHA_PLAIN * loss_plain + V3_ALPHA_WEIGHTED * loss_weighted

            probs = torch.sigmoid(logits).detach().cpu().numpy()
            y_prob.append(probs)
            y_true.append(tgts.detach().cpu().numpy())
            total_loss += float(loss_tax.item()) * tgts.shape[0]
            n += int(tgts.shape[0])

    y_true = np.concatenate(y_true, axis=0)
    y_prob = np.concatenate(y_prob, axis=0)
    return total_loss / max(n, 1), y_true, y_prob

# ── v3 tuning philosophy ──────────────────────────────────────────────────────
# v2 improved F1 but lost a bit of mAP. v3 uses:
# 1) milder weighting (sqrt imbalance, capped lower)
# 2) blended taxonomy loss: mostly plain BCE + some weighted BCE
# 3) optional op auxiliary loss with low weight
# 4) early stopping on a composite score that values both macro_f1_supported and mAP_micro

V3_EPOCHS = 6
V3_PATIENCE = 3
V3_LR = 5e-6
V3_WEIGHT_DECAY = 1e-4
V3_USE_AMP = True
V3_GRAD_CLIP = 1.0

# Blend losses to recover calibration
V3_ALPHA_PLAIN = 0.70
V3_ALPHA_WEIGHTED = 0.30

# Small auxiliary weight if operation labels exist
V3_OP_AUX_WEIGHT = 0.10

train_targets = _collect_train_targets(train_loader)
pos = train_targets.sum(axis=0)
neg = train_targets.shape[0] - pos

# milder weights than v2 to help AP/calibration
pos_weight = np.sqrt(neg / np.clip(pos, 1, None)).astype(np.float32)
pos_weight = np.clip(pos_weight, 1.0, 4.0)
pos_weight_t = torch.tensor(pos_weight, dtype=torch.float32, device=device)

print("v3 pos_weight:")
for i, w in enumerate(pos_weight.tolist()):
    print(f"  {ERROR_TYPES[i]:<26} {w:.3f}")

criterion_tax_plain = nn.BCEWithLogitsLoss()
criterion_tax_weighted = nn.BCEWithLogitsLoss(pos_weight=pos_weight_t)
criterion_op = nn.CrossEntropyLoss()

optimizer_v3 = torch.optim.AdamW(
    model.parameters(),
    lr=V3_LR,
    weight_decay=V3_WEIGHT_DECAY,
)

best_score = -1.0
best_thresholds = [0.5] * len(ERROR_TYPES)
no_improve = 0
history_v3 = []

# detect op-label availability once
op_probe_batch = next(iter(train_loader))
op_probe_ids, op_probe_key = _get_op_ids_from_batch(op_probe_batch, device)
HAS_OP_LABELS = op_probe_ids is not None
print("HAS_OP_LABELS =", HAS_OP_LABELS, "| key =", op_probe_key)

for epoch in range(1, V3_EPOCHS + 1):
    model.train()
    run_loss = 0.0
    n = 0

    for batch in train_loader:
        src, tgt, ids, mask, tgts = _batch_to_device(batch, device)
        op_ids, _ = _get_op_ids_from_batch(batch, device)

        optimizer_v3.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", enabled=(V3_USE_AMP and device.type == "cuda")):
            logits, evidence = model(src, tgt, ids, mask)
            loss_plain = criterion_tax_plain(logits, tgts)
            loss_weighted = criterion_tax_weighted(logits, tgts)
            loss_tax = V3_ALPHA_PLAIN * loss_plain + V3_ALPHA_WEIGHTED * loss_weighted

            loss = loss_tax
            if HAS_OP_LABELS and op_ids is not None and "op_logits" in evidence:
                loss_op = criterion_op(evidence["op_logits"], op_ids)
                loss = loss + V3_OP_AUX_WEIGHT * loss_op

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), V3_GRAD_CLIP)
        optimizer_v3.step()

        run_loss += float(loss.item()) * tgts.shape[0]
        n += int(tgts.shape[0])

    val_loss, y_true_v, y_prob_v = _eval_loader_v3(
        model, val_loader, criterion_tax_plain, criterion_tax_weighted, device, use_amp=V3_USE_AMP
    )
    thresholds_v = _tune_thresholds(y_true_v, y_prob_v)
    y_pred_v = (y_prob_v >= np.array(thresholds_v).reshape(1, -1)).astype(int)
    metrics_v = _compute_metrics(y_true_v, y_prob_v, y_pred_v)

    # composite score: keep F1 priority, but reward AP recovery
    score = 0.70 * metrics_v["macro_f1_supported"] + 0.30 * metrics_v["mAP_micro"]

    print(
        f"v3 Epoch {epoch:02d} | train_loss={run_loss/max(n,1):.4f} "
        f"val_loss={val_loss:.4f} | micro_f1={metrics_v['micro_f1']:.4f} "
        f"macro_f1_sup={metrics_v['macro_f1_supported']:.4f} | mAP_micro={metrics_v['mAP_micro']:.4f} "
        f"| score={score:.4f}"
    )

    history_v3.append({
        "epoch": epoch,
        "train_loss": run_loss / max(n, 1),
        "val_loss": val_loss,
        "val_score": score,
        **{f"val_{k}": v for k, v in metrics_v.items()},
    })

    if score > best_score:
        best_score = score
        best_thresholds = thresholds_v
        torch.save(model.state_dict(), V3_DIR / "best_model.pt")
        print(f"  ✓ saved new v3 best (score={score:.4f})")
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= V3_PATIENCE:
            print(f"  Early stopping v3 at epoch {epoch} (no improvement for {V3_PATIENCE} epochs)")
            break

print(f"\nV3 fine-tuning done. Best val composite score = {best_score:.4f}")

# ── load best v3 and evaluate ─────────────────────────────────────────────────
best_state = torch.load(V3_DIR / "best_model.pt", map_location=device)
model.load_state_dict(best_state)
model.to(device)
model.eval()

# primary benchmark
test_bench_loss, y_true_b, y_prob_b = _eval_loader_v3(
    model, test_loader_benchmark, criterion_tax_plain, criterion_tax_weighted, device, use_amp=V3_USE_AMP
)
y_pred_b = (y_prob_b >= np.array(best_thresholds).reshape(1, -1)).astype(int)
metrics_b = _compute_metrics(y_true_b, y_prob_b, y_pred_b)

print("\n── V3 Primary benchmark: taxonomy-positive held-out test ─────────────")
for k, v in metrics_b.items():
    print(f"  {k:<24}: {v:.4f}")

per_class_b = _per_class_df(y_true_b, y_pred_b, ERROR_TYPES)
display(per_class_b)

# full held-out
test_full_loss, y_true_f, y_prob_f = _eval_loader_v3(
    model, test_loader_full, criterion_tax_plain, criterion_tax_weighted, device, use_amp=V3_USE_AMP
)
y_pred_f = (y_prob_f >= np.array(best_thresholds).reshape(1, -1)).astype(int)
metrics_f = _compute_metrics(y_true_f, y_prob_f, y_pred_f)

print("\n── V3 Secondary eval: full held-out test rows ────────────────────────")
for k, v in metrics_f.items():
    print(f"  {k:<24}: {v:.4f}")

# save artifacts
(V3_DIR / "train_history_v3.json").write_text(json.dumps(history_v3, indent=2), encoding="utf-8")
(V3_DIR / "best_thresholds.json").write_text(json.dumps(best_thresholds, indent=2), encoding="utf-8")

metrics_out = {
    "v3_best_val_composite_score": best_score,
    "v3_benchmark_metrics": metrics_b,
    "v3_full_test_metrics": metrics_f,
    "best_thresholds": best_thresholds,
    "pos_weight": pos_weight.tolist(),
    "loss_blend": {
        "plain_bce": V3_ALPHA_PLAIN,
        "weighted_bce": V3_ALPHA_WEIGHTED,
        "op_aux_weight": V3_OP_AUX_WEIGHT if HAS_OP_LABELS else 0.0,
    },
}
(V3_DIR / "metrics.json").write_text(json.dumps(metrics_out, indent=2), encoding="utf-8")

per_class_b.to_csv(V3_DIR / "per_class_test_benchmark.csv", index=False)

with open(V3_DIR / "predictions_test_benchmark.jsonl", "w", encoding="utf-8") as f:
    for i in range(y_true_b.shape[0]):
        rec = {
            "gold_vec": y_true_b[i].astype(int).tolist(),
            "pred_vec": y_pred_b[i].astype(int).tolist(),
            "prob": [float(x) for x in y_prob_b[i].tolist()],
        }
        f.write(json.dumps(rec) + "\n")

print("\nSaved v3 artifacts:")
print(" ", V3_DIR / "best_model.pt")
print(" ", V3_DIR / "best_thresholds.json")
print(" ", V3_DIR / "metrics.json")
print(" ", V3_DIR / "per_class_test_benchmark.csv")
print(" ", V3_DIR / "predictions_test_benchmark.jsonl")

Using val_loader_benchmark for v3 selection.
Warm-starting from v2: /kaggle/working/evidence_grounded_taxonomy_eval_v2/best_model.pt
v3 pos_weight:
  Wrong Object               4.000
  Missing Object             4.000
  Extra Object               4.000
  Wrong Attribute            2.497
  Spatial Error              4.000
  Style Mismatch             2.370
  Over-editing               4.000
  Under-editing              1.000
  Artifact / Quality Issue   4.000
  Ambiguous Prompt           4.000
  Failed Removal             4.000
HAS_OP_LABELS = False | key = None
v3 Epoch 01 | train_loss=0.0275 val_loss=0.8432 | micro_f1=0.6669 macro_f1_sup=0.4559 | mAP_micro=0.6551 | score=0.5157
  ✓ saved new v3 best (score=0.5157)
v3 Epoch 02 | train_loss=0.0204 val_loss=0.8814 | micro_f1=0.6695 macro_f1_sup=0.4594 | mAP_micro=0.6586 | score=0.5192
  ✓ saved new v3 best (score=0.5192)
v3 Epoch 03 | train_loss=0.0166 val_loss=0.9192 | micro_f1=0.6675 macro_f1_sup=0.4588 | mAP_micro=0.6585 | score=0.518

,class,f1,precision,recall,support
0,Wrong Object,0.250000,0.333333,0.200000,15
1,Missing Object,0.127660,0.193548,0.095238,63
2,Extra Object,0.000000,0.000000,0.000000,0
3,Wrong Attribute,0.420601,0.662162,0.308176,159
4,Spatial Error,0.222222,0.428571,0.150000,60
5,Style Mismatch,0.415663,0.428571,0.403509,171
6,Over-editing,0.375000,1.000000,0.230769,39
7,Under-editing,0.869248,0.842776,0.897436,663
8,Artifact / Quality Issue,0.162162,0.187500,0.142857,21
9,Ambiguous Prompt,0.000000,0.000000,0.000000,0



── V3 Secondary eval: full held-out test rows ────────────────────────
  micro_f1                : 0.4835
  macro_f1                : 0.2302
  macro_f1_supported      : 0.2813
  mAP_micro               : 0.4559
  mAP_macro_supported     : 0.2362

Saved v3 artifacts:
  /kaggle/working/evidence_grounded_taxonomy_eval_v3/best_model.pt
  /kaggle/working/evidence_grounded_taxonomy_eval_v3/best_thresholds.json
  /kaggle/working/evidence_grounded_taxonomy_eval_v3/metrics.json
  /kaggle/working/evidence_grounded_taxonomy_eval_v3/per_class_test_benchmark.csv
  /kaggle/working/evidence_grounded_taxonomy_eval_v3/predictions_test_benchmark.jsonl
